In [ ]:
%pip install geopandas

In [0]:
dbutils.library.restartPython()

In [0]:
from datetime import datetime
import geopandas as gpd

In [0]:
import sys

sys.path.append("../../src")
from dataDP.utils.logger import logger
from dataDP.data_management import insert_or_update_table
from dataDP.meta.insert_process_log import insert_ingestion_log
from dataDP import get_execution_id

In [0]:
%sh
ping google.com

# Serverless computes of Databricks free have no internet access.
**Use localy src/dataDP/ingest/upload_to_databricks.py**

In [0]:
# Filex already available into the volume. Download implementation has to be done

In [ ]:
local_shp_path = "/Volumes/workspace/taxi/metadata/taxi_zones/taxi_zones.shp"
stg_table_name = "workspace.stg.coordinates"
start_time = datetime.now()

In [ ]:

try:
    # Load the shapefile

    gdf = gpd.read_file(local_shp_path)

    # Calculate centroids and extract lat/lon in decimal degrees (EPSG:4326)
    gdf_wgs84 = gdf.to_crs(epsg=4326)
    gdf_wgs84['longitude'] = gdf_wgs84.geometry.centroid.x
    gdf_wgs84['latitude'] = gdf_wgs84.geometry.centroid.y

    # Select relevant columns and convert to Spark DataFrame
    zones_data = gdf_wgs84[['LocationID', 'borough', 'zone',  'latitude', 'longitude']].copy()
    zones_df = spark.createDataFrame(zones_data)
    spark_df = spark.createDataFrame(gdf.astype(str))
    ogg_data_df = spark_df.select("LocationID", "OBJECTID", "Shape_Leng", "Shape_Area", "geometry").withColumnRenamed("LocationID", "LocationID_ref")

    new_df = zones_df.join(ogg_data_df, zones_df.LocationID == ogg_data_df.LocationID_ref, "inner") \
        .drop("LocationID_ref")
    
    insert_or_update_table(
            df=new_df,
            table_name=stg_table_name,
        )

    insert_ingestion_log(
                spark=spark,
                execution_id=get_execution_id(),
                source_system=stg_table_name,
                table_name=stg_table_name,
                status="success",
                records_processed=0,
                start_time=start_time,
                end_time=datetime.now(),
                duration_min=(datetime.now() - start_time).total_seconds() / 60,
                message=f"Staging table {stg_table_name}",
            )
except Exception as e:
    logger.error(e)
    insert_ingestion_log(
                spark=spark,
                execution_id=get_execution_id(),
                source_system=stg_table_name,
                table_name=stg_table_name,
                status="ERROR",
                records_processed=0,
                start_time=start_time,
                end_time=datetime.now(),
                duration_min=(datetime.now() - start_time).total_seconds() / 60,
                message=f"Staging table {stg_table_name} has following errors: {e}",
            )

In [ ]:

# /Volumes/workspace/taxi/metadata/taxi_zone_lookup.csv